# Notebook 16 — Baseline Hurdle Model

This notebook estimates the baseline hurdle model for journalist killings using the final monadic panel.This notebook uses the baseline feature list defined in Notebook 14 and does not add network predictors.

Model interpretation cautions:
- Iraq and Syria may dominate ODA coefficients; treat this as a possible crisis-aid confound rather than evidence of structural dependency.
- Arms effects may be driven by a small number of country-years. Prior diagnostics showed high concentration (Gini about 0.777) and density below 1%.
- ODA and economic dependency overlap is non-trivial (Jaccard about 0.55), so VIF is checked before interpreting coefficients.
- Colonial and arms PageRank are near-uniform and may add noise in the network model; this is mainly a Notebook 17 concern.
- Econ PageRank is uniform for 1993-1995 in the final panel because of the ECI coverage gap and 1-year lag offset. Document this if network features are inspected.
- Lagged predictors remove rows with missing values; exact sample loss is calculated below.
- The econ-mean robustness variant has materially smaller coverage because of about 35.6% missingness and should be interpreted separately.

In [1]:
from pathlib import Path
import pickle
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import Logit, NegativeBinomial
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings('default')
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
DATA_PATH = PROJECT_ROOT / 'data' / 'merged' / 'panel_final_1992_2024.csv'
MODEL_DIR = PROJECT_ROOT / 'outputs' / 'models'
RESULTS_DIR = PROJECT_ROOT / 'outputs' / 'results'

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Data path: {DATA_PATH.relative_to(PROJECT_ROOT)}')
print(f'Model output dir: {MODEL_DIR.relative_to(PROJECT_ROOT)}')
print(f'Results output dir: {RESULTS_DIR.relative_to(PROJECT_ROOT)}')

Data path: data\merged\panel_final_1992_2024.csv
Model output dir: outputs\models
Results output dir: outputs\results


## Load and Inspect Final Panel

In [2]:
panel = pd.read_csv(DATA_PATH)

print(f'Panel shape: {panel.shape[0]:,} rows x {panel.shape[1]:,} columns')
display(panel.head())

column_inventory = pd.DataFrame({
    'column': panel.columns,
    'dtype': [panel[c].dtype for c in panel.columns],
    'missing_n': [panel[c].isna().sum() for c in panel.columns],
    'missing_pct': [panel[c].isna().mean() * 100 for c in panel.columns],
})

display(column_inventory.sort_values(['missing_pct', 'column'], ascending=[False, True]).head(40))
column_inventory.to_csv(RESULTS_DIR / 'nb16_panel_column_missingness.csv', index=False)
print('Saved: outputs/results/nb16_panel_column_missingness.csv')

Panel shape: 6,358 rows x 38 columns


,recipient_iso3,year,arms_tiv_total,oda_total,econ_neocol_score_total,colonial_tie_flag,journalist_killings,gdp_per_capita,gdp_per_capita_log,population,population_log,armed_conflict,conflict_intensity,arms_tiv_in_strength_lag1,arms_tiv_out_strength_lag1,arms_tiv_dependency_balance_lag1,arms_tiv_in_concentration_lag1,arms_tiv_pagerank_lag1,bilateral_oda_in_strength_lag1,bilateral_oda_out_strength_lag1,bilateral_oda_dependency_balance_lag1,bilateral_oda_in_concentration_lag1,bilateral_oda_pagerank_lag1,colonial_tie_in_strength_lag1,colonial_tie_out_strength_lag1,colonial_tie_dependency_balance_lag1,colonial_tie_in_concentration_lag1,colonial_tie_pagerank_lag1,econ_neocol_score_in_strength_lag1,econ_neocol_score_out_strength_lag1,econ_neocol_score_dependency_balance_lag1,econ_neocol_score_in_concentration_lag1,econ_neocol_score_pagerank_lag1,arms_tiv_total_log,oda_total_log,arms_tiv_total_log_lag1,oda_total_log_lag1,econ_neocol_score_total_lag1
0,ABW,1992,0.0,29.85,0.0,0,0,13892.605143,9.539112,69005.0,11.141934,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,3.429137,NaN,NaN,NaN
1,ABW,1993,0.0,22.97,0.0,0,0,14700.959808,9.595668,73685.0,11.207555,0.0,0.0,0.0,0.0,0.0,0.0,0.003989,3.429137,0.0,1.488205,1.0,0.004430,0.0,0.0,0.0,0.0,0.004721,0.0,0.0,0.0,0.0,0.004878,0.0,3.176803,0.0,3.429137,0.0
2,ABW,1994,0.0,15.93,0.0,0,0,16055.287787,9.683794,77595.0,11.259258,0.0,0.0,0.0,0.0,0.0,0.0,0.003765,3.176803,0.0,1.429546,1.0,0.004371,0.0,0.0,0.0,0.0,0.004634,0.0,0.0,0.0,0.0,0.004785,0.0,2.829087,0.0,3.176803,0.0
3,ABW,1995,0.0,17.98,0.0,0,0,16548.717387,9.714064,79805.0,11.287341,0.0,0.0,0.0,0.0,0.0,0.0,0.003786,2.829087,0.0,1.342626,1.0,0.004361,0.0,0.0,0.0,0.0,0.004655,0.0,0.0,0.0,0.0,0.004808,0.0,2.943386,0.0,2.829087,0.0
4,ABW,1996,0.0,20.33,0.0,0,0,16620.954556,9.718420,83021.0,11.326849,0.0,0.0,0.0,0.0,0.0,0.0,0.003755,2.943386,0.0,1.372040,1.0,0.004389,0.0,0.0,0.0,0.0,0.004655,0.0,0.0,0.0,0.0,0.004126,0.0,3.060115,0.0,2.943386,0.0


,column,dtype,missing_n,missing_pct
7,gdp_per_capita,float64,404,6.354199
8,gdp_per_capita_log,float64,404,6.354199
9,population,float64,268,4.215162
10,population_log,float64,268,4.215162
11,armed_conflict,float64,235,3.696131
12,conflict_intensity,float64,235,3.696131
35,arms_tiv_total_log_lag1,float64,213,3.350110
37,econ_neocol_score_total_lag1,float64,213,3.350110
36,oda_total_log_lag1,float64,213,3.350110
15,arms_tiv_dependency_balance_lag1,float64,208,3.271469


Saved: outputs/results/nb16_panel_column_missingness.csv


## Baseline Features from Notebook 14

In [4]:
baseline_features = [
    'arms_tiv_total_log_lag1',
    'oda_total_log_lag1',
    'econ_neocol_score_total_lag1',
    'colonial_tie_flag',
    'gdp_per_capita_log',
    'population_log',
    'armed_conflict',
    'conflict_intensity',
]

outcome = 'journalist_killings'
required_columns = baseline_features + [outcome]
missing_features = [c for c in required_columns if c not in panel.columns]

if missing_features:
    raise KeyError(f'Missing required columns: {missing_features}')

feature_missingness = (
    panel[required_columns]
    .isna()
    .agg(['sum', 'mean'])
    .T
    .rename(columns={'sum': 'missing_n', 'mean': 'missing_pct'})
)
feature_missingness['missing_pct'] *= 100

print('Baseline features:')
for feature in baseline_features:
    print(f'- {feature}')

display(feature_missingness)
feature_missingness.to_csv(RESULTS_DIR / 'nb16_baseline_feature_missingness.csv')
print('Saved: outputs/results/nb16_baseline_feature_missingness.csv')

Baseline features:
- arms_tiv_total_log_lag1
- oda_total_log_lag1
- econ_neocol_score_total_lag1
- colonial_tie_flag
- gdp_per_capita_log
- population_log
- armed_conflict
- conflict_intensity


,missing_n,missing_pct
arms_tiv_total_log_lag1,213.0,3.350110
oda_total_log_lag1,213.0,3.350110
econ_neocol_score_total_lag1,213.0,3.350110
colonial_tie_flag,0.0,0.000000
gdp_per_capita_log,404.0,6.354199
population_log,268.0,4.215162
armed_conflict,235.0,3.696131
conflict_intensity,235.0,3.696131
journalist_killings,0.0,0.000000


Saved: outputs/results/nb16_baseline_feature_missingness.csv


## Analysis Sample

Rows with missing values in the baseline features or the count outcome are removed. This is expected to drop a small share of the final panel because of lagged predictors; the exact loss is reported here rather than assumed.

In [5]:
before_n = len(panel)
model_df = panel.dropna(subset=required_columns).copy()
after_n = len(model_df)
dropped_n = before_n - after_n
dropped_pct = dropped_n / before_n * 100

model_df['killing_any'] = (model_df[outcome] > 0).astype(int)

sample_report = pd.DataFrame([{
    'rows_before': before_n,
    'rows_after': after_n,
    'rows_dropped': dropped_n,
    'pct_dropped': dropped_pct,
    'nonzero_count_rows': int((model_df[outcome] > 0).sum()),
    'zero_count_rows': int((model_df[outcome] == 0).sum()),
}])

display(sample_report)
sample_report.to_csv(RESULTS_DIR / 'nb16_sample_loss.csv', index=False)
print('Saved: outputs/results/nb16_sample_loss.csv')

,rows_before,rows_after,rows_dropped,pct_dropped,nonzero_count_rows,zero_count_rows
0,6358,5765,593,9.326832,687,5078


Saved: outputs/results/nb16_sample_loss.csv


## Helper Functions

In [6]:
def add_significance(p_value):
    if pd.isna(p_value):
        return ''
    if p_value < 0.001:
        return '***'
    if p_value < 0.01:
        return '**'
    if p_value < 0.05:
        return '*'
    if p_value < 0.1:
        return '.'
    return ''


def model_table(result):
    table = pd.DataFrame({
        'term': result.params.index,
        'coef': result.params.values,
        'std_err': result.bse.values,
        'z': result.tvalues.values,
        'p_value': result.pvalues.values,
    })
    table['significance'] = table['p_value'].map(add_significance)
    return table


def model_metrics(result, model_name):
    return pd.DataFrame([{
        'model': model_name,
        'nobs': int(result.nobs),
        'aic': result.aic,
        'log_likelihood': result.llf,
        'pseudo_r2': getattr(result, 'prsquared', np.nan),
    }])


def gini(values):
    x = pd.Series(values).dropna().astype(float)
    x = x[x >= 0].sort_values().to_numpy()
    if len(x) == 0 or np.isclose(x.sum(), 0):
        return np.nan
    n = len(x)
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * x) / (n * np.sum(x))) - ((n + 1) / n)

## Part A — Logistic Regression

This model estimates which baseline predictors are associated with any journalist killing in a country-year.

In [7]:
X_logit = sm.add_constant(model_df[baseline_features], has_constant='add')
y_logit = model_df['killing_any']

logit_model = Logit(y_logit, X_logit)
logit_result = logit_model.fit(maxiter=200, disp=False)

logit_table = model_table(logit_result)
logit_metrics = model_metrics(logit_result, 'logit_any_killing')

display(logit_table)
display(logit_metrics)
print(logit_result.summary())

with open(MODEL_DIR / 'nb16_logit_any_killing.pkl', 'wb') as f:
    pickle.dump(logit_result, f)
logit_table.to_csv(RESULTS_DIR / 'nb16_logit_any_killing_coefficients.csv', index=False)
logit_metrics.to_csv(RESULTS_DIR / 'nb16_logit_any_killing_metrics.csv', index=False)

print('Saved: outputs/models/nb16_logit_any_killing.pkl')
print('Saved: outputs/results/nb16_logit_any_killing_coefficients.csv')
print('Saved: outputs/results/nb16_logit_any_killing_metrics.csv')

,term,coef,std_err,z,p_value,significance
0,const,-14.055621,0.789139,-17.811327,5.772500e-71,***
1,arms_tiv_total_log_lag1,-0.078649,0.027199,-2.891635,3.832426e-03,**
2,oda_total_log_lag1,0.181880,0.030549,5.953753,2.620615e-09,***
3,econ_neocol_score_total_lag1,-0.027873,0.020166,-1.382149,1.669259e-01,
4,colonial_tie_flag,0.487632,0.116938,4.169993,3.046094e-05,***
5,gdp_per_capita_log,0.215029,0.048203,4.460944,8.159958e-06,***
6,population_log,0.552139,0.040984,13.472017,2.285369e-41,***
7,armed_conflict,-0.475067,0.238706,-1.990178,4.657136e-02,*
8,conflict_intensity,1.542452,0.171935,8.971155,2.934215e-19,***


,model,nobs,aic,log_likelihood,pseudo_r2
0,logit_any_killing,5765,3097.155926,-1539.577963,0.268866


                           Logit Regression Results                           
Dep. Variable:            killing_any   No. Observations:                 5765
Model:                          Logit   Df Residuals:                     5756
Method:                           MLE   Df Model:                            8
Date:                Thu, 14 May 2026   Pseudo R-squ.:                  0.2689
Time:                        17:32:24   Log-Likelihood:                -1539.6
converged:                       True   LL-Null:                       -2105.7
Covariance Type:            nonrobust   LLR p-value:                3.999e-239
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const                          -14.0556      0.789    -17.811      0.000     -15.602     -12.509
arms_tiv_total_log_lag1         -0.0786      0.027     -2.892      0.004     

## Part B — Negative Binomial on Non-Zero Counts

This model estimates the number of killings conditional on at least one killing. It is fit only on non-zero journalist-killing country-years.

In [8]:
positive_df = model_df.loc[model_df[outcome] > 0].copy()

X_nb = sm.add_constant(positive_df[baseline_features], has_constant='add')
y_nb = positive_df[outcome]

nb_model = NegativeBinomial(y_nb, X_nb)
nb_result = nb_model.fit(maxiter=300, disp=False)

nb_table = model_table(nb_result)
nb_metrics = model_metrics(nb_result, 'negative_binomial_positive_counts')

display(nb_table)
display(nb_metrics)
print(nb_result.summary())

with open(MODEL_DIR / 'nb16_negative_binomial_positive_counts.pkl', 'wb') as f:
    pickle.dump(nb_result, f)
nb_table.to_csv(RESULTS_DIR / 'nb16_negative_binomial_positive_counts_coefficients.csv', index=False)
nb_metrics.to_csv(RESULTS_DIR / 'nb16_negative_binomial_positive_counts_metrics.csv', index=False)

print('Saved: outputs/models/nb16_negative_binomial_positive_counts.pkl')
print('Saved: outputs/results/nb16_negative_binomial_positive_counts_coefficients.csv')
print('Saved: outputs/results/nb16_negative_binomial_positive_counts_metrics.csv')

,term,coef,std_err,z,p_value,significance
0,const,-2.839931,0.642434,-4.420583,9.843498e-06,***
1,arms_tiv_total_log_lag1,-0.032561,0.020715,-1.571876,1.159793e-01,
2,oda_total_log_lag1,0.057552,0.023437,2.455652,1.406292e-02,*
3,econ_neocol_score_total_lag1,-0.031837,0.016294,-1.953950,5.070709e-02,.
4,colonial_tie_flag,0.307667,0.100372,3.065256,2.174840e-03,**
5,gdp_per_capita_log,0.245565,0.036874,6.659530,2.747056e-11,***
6,population_log,0.070898,0.034239,2.070696,3.838721e-02,*
7,armed_conflict,-0.965145,0.158912,-6.073439,1.251999e-09,***
8,conflict_intensity,1.245413,0.096903,12.852175,8.362557e-38,***
9,alpha,0.463164,0.036532,12.678149,7.815547e-37,***


,model,nobs,aic,log_likelihood,pseudo_r2
0,negative_binomial_positive_counts,687,2899.301819,-1439.65091,0.098355


                      NegativeBinomial Regression Results                      
Dep. Variable:     journalist_killings   No. Observations:                  687
Model:                NegativeBinomial   Df Residuals:                      678
Method:                            MLE   Df Model:                            8
Date:                 Thu, 14 May 2026   Pseudo R-squ.:                 0.09836
Time:                         17:32:39   Log-Likelihood:                -1439.7
converged:                        True   LL-Null:                       -1596.7
Covariance Type:             nonrobust   LLR p-value:                 4.124e-63
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const                           -2.8399      0.642     -4.421      0.000      -4.099      -1.581
arms_tiv_total_log_lag1         -0.0326      0.021     -1.572      0.

## VIF Diagnostics

VIF is checked for all baseline predictors, with particular attention to ODA and economic dependency because prior dyadic overlap diagnostics found ODA/econ Jaccard overlap around 0.55.

In [9]:
vif_X = sm.add_constant(model_df[baseline_features], has_constant='add')
vif_rows = []

for i, col in enumerate(vif_X.columns):
    if col == 'const':
        continue
    vif_rows.append({
        'feature': col,
        'vif': variance_inflation_factor(vif_X.values, i),
    })

vif_table = pd.DataFrame(vif_rows).sort_values('vif', ascending=False)
display(vif_table)
vif_table.to_csv(RESULTS_DIR / 'nb16_vif_baseline_features.csv', index=False)
print('Saved: outputs/results/nb16_vif_baseline_features.csv')

,feature,vif
6,armed_conflict,8.223534
7,conflict_intensity,8.055302
1,oda_total_log_lag1,2.483327
4,gdp_per_capita_log,2.478368
0,arms_tiv_total_log_lag1,2.295922
5,population_log,2.234087
2,econ_neocol_score_total_lag1,1.444244
3,colonial_tie_flag,1.353028


Saved: outputs/results/nb16_vif_baseline_features.csv


## Crisis-Aid Sensitivity for ODA

Iraq and Syria are inspected because high-casualty crisis country-years can create an ODA coefficient that reflects emergency or post-conflict aid rather than structural dependency.

In [10]:
country_col = 'recipient_iso3' if 'recipient_iso3' in model_df.columns else None
crisis_codes = ['IRQ', 'SYR']

if country_col is None:
    crisis_summary = pd.DataFrame([{'note': 'recipient_iso3 column not available; crisis-country sensitivity not run.'}])
    oda_sensitivity = pd.DataFrame()
else:
    crisis_mask = model_df[country_col].isin(crisis_codes)
    crisis_summary = (
        model_df.assign(crisis_country_year=crisis_mask)
        .groupby('crisis_country_year')
        .agg(
            rows=(outcome, 'size'),
            killing_rows=('killing_any', 'sum'),
            killings_total=(outcome, 'sum'),
            mean_killings=(outcome, 'mean'),
            mean_oda_lag1=('oda_total_log_lag1', 'mean'),
            max_oda_lag1=('oda_total_log_lag1', 'max'),
        )
        .reset_index()
    )

    non_crisis_df = model_df.loc[~crisis_mask].copy()
    non_crisis_positive_df = non_crisis_df.loc[non_crisis_df[outcome] > 0].copy()

    sensitivity_rows = []
    if non_crisis_df['killing_any'].nunique() == 2:
        logit_nc = Logit(
            non_crisis_df['killing_any'],
            sm.add_constant(non_crisis_df[baseline_features], has_constant='add'),
        ).fit(maxiter=200, disp=False)
        sensitivity_rows.append({
            'model': 'logit_any_killing',
            'sample': 'all_baseline_rows',
            'oda_coef': logit_result.params.get('oda_total_log_lag1'),
            'oda_p_value': logit_result.pvalues.get('oda_total_log_lag1'),
            'nobs': int(logit_result.nobs),
        })
        sensitivity_rows.append({
            'model': 'logit_any_killing',
            'sample': 'excluding_irq_syr',
            'oda_coef': logit_nc.params.get('oda_total_log_lag1'),
            'oda_p_value': logit_nc.pvalues.get('oda_total_log_lag1'),
            'nobs': int(logit_nc.nobs),
        })

    if len(non_crisis_positive_df) > len(baseline_features) + 2:
        nb_nc = NegativeBinomial(
            non_crisis_positive_df[outcome],
            sm.add_constant(non_crisis_positive_df[baseline_features], has_constant='add'),
        ).fit(maxiter=300, disp=False)
        sensitivity_rows.append({
            'model': 'negative_binomial_positive_counts',
            'sample': 'all_positive_rows',
            'oda_coef': nb_result.params.get('oda_total_log_lag1'),
            'oda_p_value': nb_result.pvalues.get('oda_total_log_lag1'),
            'nobs': int(nb_result.nobs),
        })
        sensitivity_rows.append({
            'model': 'negative_binomial_positive_counts',
            'sample': 'excluding_irq_syr',
            'oda_coef': nb_nc.params.get('oda_total_log_lag1'),
            'oda_p_value': nb_nc.pvalues.get('oda_total_log_lag1'),
            'nobs': int(nb_nc.nobs),
        })

    oda_sensitivity = pd.DataFrame(sensitivity_rows)

display(crisis_summary)
display(oda_sensitivity)
crisis_summary.to_csv(RESULTS_DIR / 'nb16_irq_syr_crisis_summary.csv', index=False)
oda_sensitivity.to_csv(RESULTS_DIR / 'nb16_oda_irq_syr_sensitivity.csv', index=False)
print('Saved: outputs/results/nb16_irq_syr_crisis_summary.csv')
print('Saved: outputs/results/nb16_oda_irq_syr_sensitivity.csv')

,crisis_country_year,rows,killing_rows,killings_total,mean_killings,mean_oda_lag1,max_oda_lag1
0,False,5703,657,1828,0.320533,3.757675,9.812317
1,True,62,30,440,7.096774,6.460704,9.999199


,model,sample,oda_coef,oda_p_value,nobs
0,logit_any_killing,all_baseline_rows,0.181880,2.620615e-09,5765
1,logit_any_killing,excluding_irq_syr,0.165010,1.229062e-07,5703
2,negative_binomial_positive_counts,all_positive_rows,0.057552,1.406292e-02,687
3,negative_binomial_positive_counts,excluding_irq_syr,0.006274,7.999656e-01,657


Saved: outputs/results/nb16_irq_syr_crisis_summary.csv
Saved: outputs/results/nb16_oda_irq_syr_sensitivity.csv


## Arms Distribution and Concentration

The arms variables are inspected for skewness, sparsity, and concentration. This matters because even statistically visible arms coefficients may be driven by a small number of country-years.

In [11]:
arms_candidates = [
    'arms_tiv_total',
    'arms_tiv_total_log',
    'arms_tiv_total_log_lag1',
    'arms_tiv_in_strength_lag1',
    'arms_tiv_pagerank_lag1',
]
arms_cols = [c for c in arms_candidates if c in panel.columns]

arms_rows = []
for col in arms_cols:
    s = panel[col].dropna().astype(float)
    positive = s[s > 0]
    arms_rows.append({
        'variable': col,
        'n': int(s.size),
        'missing_pct': panel[col].isna().mean() * 100,
        'zero_share_pct': (s.eq(0).mean() * 100) if s.size else np.nan,
        'positive_n': int(positive.size),
        'positive_share_pct': (positive.size / s.size * 100) if s.size else np.nan,
        'mean': s.mean() if s.size else np.nan,
        'median': s.median() if s.size else np.nan,
        'std': s.std() if s.size else np.nan,
        'skew': s.skew() if s.size else np.nan,
        'p95': s.quantile(0.95) if s.size else np.nan,
        'p99': s.quantile(0.99) if s.size else np.nan,
        'max': s.max() if s.size else np.nan,
        'gini_nonnegative': gini(s),
        'top_1pct_share': (s.nlargest(max(1, int(np.ceil(0.01 * len(s))))).sum() / s.sum()) if s.sum() > 0 else np.nan,
        'top_5pct_share': (s.nlargest(max(1, int(np.ceil(0.05 * len(s))))).sum() / s.sum()) if s.sum() > 0 else np.nan,
    })

arms_diagnostics = pd.DataFrame(arms_rows)
display(arms_diagnostics)
arms_diagnostics.to_csv(RESULTS_DIR / 'nb16_arms_distribution_diagnostics.csv', index=False)
print('Saved: outputs/results/nb16_arms_distribution_diagnostics.csv')

,variable,n,missing_pct,zero_share_pct,positive_n,positive_share_pct,mean,median,std,skew,p95,p99,max,gini_nonnegative,top_1pct_share,top_5pct_share
0,arms_tiv_total,6358,0.000000,43.236867,3609,56.763133,131.945771,3.000000,376.327875,5.598435,757.117000,1884.002400,5271.930000,0.859075,0.213211,0.552184
1,arms_tiv_total_log,6358,0.000000,43.236867,3609,56.763133,2.209530,1.386294,2.421501,0.627530,6.630838,7.541679,8.570341,0.595414,0.035998,0.162395
2,arms_tiv_total_log_lag1,6145,3.350110,43.238405,3488,56.761595,2.208370,1.386294,2.421642,0.628514,6.628628,7.555458,8.570341,0.595687,0.036079,0.162809
3,arms_tiv_in_strength_lag1,6150,3.271469,44.113821,3437,55.886179,5.371508,1.311028,8.510594,2.507320,23.558189,37.171041,108.991998,0.718104,0.084739,0.296625
4,arms_tiv_pagerank_lag1,6150,3.271469,0.000000,6150,100.000000,0.005075,0.004139,0.002606,6.749906,0.009511,0.015597,0.066294,0.184746,0.041502,0.134193


Saved: outputs/results/nb16_arms_distribution_diagnostics.csv


## Output Inventory

In [12]:
expected_outputs = [
    MODEL_DIR / 'nb16_logit_any_killing.pkl',
    MODEL_DIR / 'nb16_negative_binomial_positive_counts.pkl',
    RESULTS_DIR / 'nb16_panel_column_missingness.csv',
    RESULTS_DIR / 'nb16_baseline_feature_missingness.csv',
    RESULTS_DIR / 'nb16_sample_loss.csv',
    RESULTS_DIR / 'nb16_logit_any_killing_coefficients.csv',
    RESULTS_DIR / 'nb16_logit_any_killing_metrics.csv',
    RESULTS_DIR / 'nb16_negative_binomial_positive_counts_coefficients.csv',
    RESULTS_DIR / 'nb16_negative_binomial_positive_counts_metrics.csv',
    RESULTS_DIR / 'nb16_vif_baseline_features.csv',
    RESULTS_DIR / 'nb16_irq_syr_crisis_summary.csv',
    RESULTS_DIR / 'nb16_oda_irq_syr_sensitivity.csv',
    RESULTS_DIR / 'nb16_arms_distribution_diagnostics.csv',
]

output_inventory = pd.DataFrame({
    'path': [str(p.relative_to(PROJECT_ROOT)) for p in expected_outputs],
    'exists': [p.exists() for p in expected_outputs],
})
display(output_inventory)

,path,exists
0,outputs\models\nb16_logit_any_killing.pkl,True
1,outputs\models\nb16_negative_binomial_positive...,True
2,outputs\results\nb16_panel_column_missingness.csv,True
3,outputs\results\nb16_baseline_feature_missingn...,True
4,outputs\results\nb16_sample_loss.csv,True
5,outputs\results\nb16_logit_any_killing_coeffic...,True
6,outputs\results\nb16_logit_any_killing_metrics...,True
7,outputs\results\nb16_negative_binomial_positiv...,True
8,outputs\results\nb16_negative_binomial_positiv...,True
9,outputs\results\nb16_vif_baseline_features.csv,True


## Notes for Interpretation

Interpret signs, magnitudes, and p-values conservatively. Null or weak findings should be reported as such. ODA effects should be read alongside the Iraq/Syria sensitivity table, arms effects alongside the distribution diagnostics, and ODA/econ coefficients alongside VIF. Network PageRank concerns are documented here for continuity but are mainly relevant to Notebook 17.